# 🔄 Python Recursion Fundamentals — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Recursion is a function that looks in a mirror and sees a smaller version of itself. Every call adds a frame to the call stack — like stacking plates. The base case is the bottom plate that doesn't need any more work. Return values bubble back up through each plate in reverse order, each one using the result below it to compute its own. When the stack gets too deep, Python panics and throws RecursionError.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [Visual Model — Call Stack as Stack of Frames](#1) |
| 2 | [Anatomy of Recursion](#2) |
| 3 | [Decision Map — Recursion Patterns](#3) |
| 4 | [Pattern 1: Linear Recursion — fib three ways](#4) |
| 5 | [Pattern 2: Tree Recursion — branching calls](#5) |
| 6 | [Pattern 3: Tail Recursion & Python's Limit](#6) |
| 7 | [Pattern 4: Memoization (top-down DP)](#7) |
| 8 | [Pattern 5: Backtracking = Recursion + Undo](#8) |
| 9 | [Pattern 6: Divide and Conquer](#9) |
| 10 | [Common Pitfalls](#10) |
| 11 | [LC 206 — Reverse Linked List (recursive)](#11) |
| 12 | [LC 104 — Max Depth of Binary Tree](#12) |
| 13 | [Full Decision Map](#13) |
| 14 | [Interview Cheat Sheet](#14) |


<a id='1'></a>

## 1. Visual Model — Call Stack as Stack of Frames

```
fib(4) — call stack grows DOWN, return values bubble UP

  fib(4)          ← top of call, waiting for fib(3) + fib(2)
    fib(3)        ← waiting for fib(2) + fib(1)
      fib(2)      ← waiting for fib(1) + fib(0)
        fib(1)    ← BASE CASE → returns 1
        fib(0)    ← BASE CASE → returns 0
      ← fib(2) = 1+0 = 1
      fib(1)      ← BASE CASE → returns 1
    ← fib(3) = 1+1 = 2
    fib(2)        ← already computed, but tree recursion calls again!
      fib(1)      ← BASE CASE → returns 1
      fib(0)      ← BASE CASE → returns 0
    ← fib(2) = 1
  ← fib(4) = 2+1 = 3

STACK FRAME = one function call = local variables + return address + return value
  Each frame pushed onto the OS stack when called, popped when it returns.
  Stack depth = deepest recursion level = O(n) for linear, O(log n) for binary recursion.

WITHOUT MEMOIZATION: fib(4) makes 9 calls, fib(30) makes 2.7 million calls.
WITH MEMOIZATION:    fib(4) makes 7 calls, fib(30) makes 59 calls.
```


<a id='2'></a>

## 2. Anatomy of Recursion

```
EVERY RECURSIVE FUNCTION HAS EXACTLY TWO PARTS:

  1. BASE CASE   — stops the recursion, returns directly
                   "the bottom plate on the stack"

  2. RECURSIVE CASE — calls itself with a SMALLER input
                      "do a little work, then trust the smaller version"

TEMPLATE:
  def solve(problem):
      if is_base_case(problem):          # ← MUST come first
          return base_case_result

      smaller = reduce(problem)          # make it smaller — REQUIRED
      sub_answer = solve(smaller)        # trust the recursion
      return combine(sub_answer, problem) # build answer from sub-answer

THREE QUESTIONS TO ASK:
  1. What is the base case? (smallest valid input)
  2. What is one unit of work? (what does one call do?)
  3. How does the result combine? (what do I do with the return value?)

RECURSION PITFALLS:
  ❌ Missing base case → infinite recursion → RecursionError
  ❌ Not reducing input → same size call → infinite recursion
  ❌ Wrong return → losing the return value → gets None back
  ❌ Mutating a shared list → side effects corrupt sibling calls
```


<a id='3'></a>

## 3. Decision Map — Recursion Patterns

```
PATTERN              SHAPE                    EXAMPLES
─────────────────────────────────────────────────────────────────────────
Linear recursion     one recursive call       reverse string, power(n)
Tree recursion       two+ recursive calls     fibonacci, tree traversal
Memoized recursion   tree + cache             DP top-down, climb stairs
Tail recursion       last act is the call     factorial(n, acc)
Divide & conquer     split + combine          merge sort, binary search
Backtracking         recurse + undo           permutations, subsets, N-queens
Mutual recursion     f() calls g(), g() calls f()  even/odd number check

WHEN TO USE RECURSION:
  ✅ Tree / graph traversal (DFS)
  ✅ Divide & conquer (merge sort, binary search)
  ✅ Backtracking (generate all paths/subsets/permutations)
  ✅ Problem has "if I solve n-1, I can solve n" structure
  ✅ Linked list operations (reverse, detect cycle)

WHEN TO USE ITERATION INSTEAD:
  ❌ Python recursion limit (sys.getrecursionlimit() = 1000 default)
  ❌ Very deep recursion (n > 1000 for linear recursion)
  ❌ Tail recursion (Python doesn't optimize it — use a loop or accumulator)
  ❌ Simple loops (no need to pay recursion overhead)
```


<a id='4'></a>

## 4. 🧩 Pattern 1: Linear Recursion — fib Three Ways

---

```
PROBLEM:  Compute the n-th Fibonacci number (fib(0)=0, fib(1)=1, fib(n)=fib(n-1)+fib(n-2)).

THREE APPROACHES:
  1. Naive recursion — O(2^n) time, O(n) space — exponential, unusable for n>30
  2. Memoized recursion — O(n) time, O(n) space — cache results
  3. Iterative — O(n) time, O(1) space — no stack, always preferred in Python

SLOW MOTION TRACE (naive) fib(4):
  call tree:
  fib(4)
  ├── fib(3)
  │   ├── fib(2) ← computed
  │   └── fib(1) ← base case
  └── fib(2)     ← RECOMPUTED — wasted work!
      ├── fib(1) ← base case
      └── fib(0) ← base case

SLOW MOTION TRACE (iterative):
  n=4:  a=0,b=1
  i=1:  a=1,b=1   (b,a+b) = (1,1)
  i=2:  a=1,b=2
  i=3:  a=2,b=3
  i=4:  a=3,b=5 → return a=3... wait, fib(4)=3 ✓

KEY INSIGHT: Python has no TCO (tail call optimization).
             Tail-recursive fib still uses O(n) stack space.
             Use the iterative version for any n > ~900.

TIME / SPACE:
  Naive:   O(2^n) / O(n stack)
  Memoized: O(n) / O(n cache + n stack)
  Iterative: O(n) / O(1)
```


In [ ]:
import sys
from functools import lru_cache

# ── 1. Naive recursion — O(2^n) — DO NOT use for large n ─────────────────────
def fib_naive(n: int) -> int:
    """
    Fibonacci — naive recursion. Exponential due to repeated sub-problems.
    Time: O(2^n)  Space: O(n) stack
    """
    if n <= 1:
        return n                         # base case: fib(0)=0, fib(1)=1
    return fib_naive(n - 1) + fib_naive(n - 2)  # tree recursion — recomputes everything

# ── 2. Memoized recursion — O(n) — top-down DP ───────────────────────────────
@lru_cache(maxsize=None)
def fib_memo(n: int) -> int:
    """
    Fibonacci with @lru_cache. First call computes, all future calls hit cache.
    Time: O(n)  Space: O(n) for cache + O(n) stack
    """
    if n <= 1:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)  # cached: each unique n computed once

# ── 3. Iterative — O(n) time, O(1) space — PREFERRED in Python ───────────────
def fib_iter(n: int) -> int:
    """
    LC 509 — Fibonacci Number (iterative).
    Approach: rolling two variables — no stack, no cache.
    Args:
        n (int): non-negative integer.
    Returns:
        int: n-th Fibonacci number.
    Time:  O(n)  — single loop
    Space: O(1)  — two variables
    """
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b       # advance: old b becomes new a, new b = sum
    return b

# Slow motion on n=5:
# a=0,b=1
# iter 1: a=1,b=1
# iter 2: a=1,b=2
# iter 3: a=2,b=3
# iter 4: a=3,b=5 → return 5 ✓  fib(5)=5

# ── Compare outputs ───────────────────────────────────────────────────────────
print("Fibonacci comparison (n=0..10):")
for n in range(11):
    naive = fib_naive(n)
    memo  = fib_memo(n)
    itr   = fib_iter(n)
    match = "✓" if naive == memo == itr else "✗ MISMATCH"
    print(f"  fib({n:2d}) = {itr:4d}  {match}")

# ── Recursion limit demo ──────────────────────────────────────────────────────
print(f"\nDefault recursion limit: {sys.getrecursionlimit()}")
print("fib_naive(30) =", fib_naive(30))         # slow but works
print("fib_iter(1000) =", fib_iter(1000))        # fast, no limit issue

# fib_naive(1000) would hit RecursionError — don't run it

# Simplicity and clarity is Gold


<a id='5'></a>

## 5. 🧩 Pattern 2: Tree Recursion — LC 104, 226

---

```
PROBLEM:  Problems on binary trees naturally decompose into two sub-problems:
          "solve left subtree" + "solve right subtree" + "combine".

APPROACH: Every tree recursive function follows:
  result = combine(f(node.left), f(node.right), node.val)
  Base case: if node is None, return some sentinel (0, True, [], etc.)

SLOW MOTION TRACE — max depth of tree:
     1
    / \
   2   3
  /
 4

  max_depth(1):
    max_depth(2):
      max_depth(4):
        max_depth(None) → 0
        max_depth(None) → 0
      ← 1 + max(0,0) = 1
      max_depth(None) → 0
    ← 1 + max(1,0) = 2
    max_depth(3):
      max_depth(None) → 0
      max_depth(None) → 0
    ← 1 + max(0,0) = 1
  ← 1 + max(2,1) = 3  ✓

KEY INSIGHT: The base case for tree problems is almost always "if not node: return X".
             X is the "empty tree" answer (0 for depth, True for valid, [] for paths).

TIME / SPACE:
  Time:  O(n) — every node visited exactly once
  Space: O(h) — call stack depth = tree height (O(log n) balanced, O(n) skewed)
```


In [ ]:
from typing import Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def maxDepth(root: Optional[TreeNode]) -> int:
    """
    LC 104 — Maximum Depth of Binary Tree.
    Approach: tree recursion — depth = 1 + max(left_depth, right_depth).
    Args:
        root (Optional[TreeNode]): root of binary tree.
    Returns:
        int: maximum depth (empty tree = 0).
    Time:  O(n) — visit every node once
    Space: O(h) — call stack = tree height
    """
    if not root:
        return 0                          # base case: empty subtree has depth 0
    left_depth  = maxDepth(root.left)     # trust recursion for left subtree
    right_depth = maxDepth(root.right)    # trust recursion for right subtree
    return 1 + max(left_depth, right_depth)  # this node + deeper child

def invertTree(root: Optional[TreeNode]) -> Optional[TreeNode]:
    """
    LC 226 — Invert Binary Tree.
    Approach: swap children at every node, recurse both sides.
    Args:
        root (Optional[TreeNode]): root of binary tree.
    Returns:
        Optional[TreeNode]: root of inverted tree.
    Time:  O(n)  Space: O(h)
    """
    if not root:
        return None                       # base case: nothing to invert
    root.left, root.right = root.right, root.left   # swap children
    invertTree(root.left)                 # invert left subtree (was right)
    invertTree(root.right)                # invert right subtree (was left)
    return root

# ── Build test tree ───────────────────────────────────────────────────────────
#      1
#     / \
#    2   3
#   /
#  4
root = TreeNode(1)
root.left  = TreeNode(2, TreeNode(4), None)
root.right = TreeNode(3)

print(f"maxDepth = {maxDepth(root)}")     # 3

def tree_to_list(root):
    """BFS to list for verification."""
    if not root: return []
    from collections import deque
    result, q = [], deque([root])
    while q:
        node = q.popleft()
        result.append(node.val if node else None)
        if node:
            q.append(node.left)
            q.append(node.right)
    while result and result[-1] is None:
        result.pop()
    return result

print(f"before invert: {tree_to_list(root)}")   # [1,2,3,4]
invertTree(root)
print(f"after invert:  {tree_to_list(root)}")   # [1,3,2,None,None,None,4]

# Simplicity and clarity is Gold


<a id='6'></a>

## 6. 🧩 Pattern 3: Tail Recursion & Python's Limit

---

```
PROBLEM:  Python does NOT perform tail call optimization (TCO).
          A tail-recursive function still grows the call stack O(n) deep.

TAIL RECURSION: the recursive call is the LAST operation — no work after it returns.
  def factorial_tail(n, acc=1):    # acc carries the running product
      if n <= 1: return acc
      return factorial_tail(n-1, n*acc)  # last act = recursive call

  In TCO languages (Scala, Haskell), this reuses the same stack frame.
  In Python: it still pushes n frames. factorial_tail(5000) raises RecursionError.

PYTHON RECURSION LIMIT:
  sys.getrecursionlimit() = 1000 by default
  sys.setrecursionlimit(10000)  ← can increase, but risky (OS stack overflow)

RULE: any linear recursion in Python that might reach n>900 → convert to a loop.

ITERATIVE EQUIVALENT of tail recursion:
  def factorial_iter(n):
      acc = 1
      while n > 1:
          acc *= n
          n -= 1
      return acc
  ← O(n) time, O(1) space — always prefer this in Python

KEY INSIGHT: Tail recursion is a coding style, not a Python optimization.
             The iterative version is strictly better in Python.
```


In [ ]:
import sys

# ── Tail recursive factorial ──────────────────────────────────────────────────
def factorial_tail(n: int, acc: int = 1) -> int:
    """
    Factorial — tail recursive style (accumulator pattern).
    NOTE: Python does NOT optimize this. Stack still grows O(n) deep.
    Args:
        n (int): non-negative integer.
        acc (int): accumulator carrying running product.
    Returns:
        int: n!
    Time:  O(n)  Space: O(n) stack — NOT O(1), no TCO in Python
    """
    if n <= 1:
        return acc                     # base case: all done, return accumulated product
    return factorial_tail(n - 1, n * acc)  # tail call: acc carries n * ... * current

# ── Iterative factorial — PREFERRED in Python ─────────────────────────────────
def factorial_iter(n: int) -> int:
    """
    Factorial — iterative. O(n) time, O(1) space.
    This is what the tail recursive version compiles to in TCO languages.
    """
    acc = 1
    while n > 1:
        acc *= n
        n  -= 1
    return acc

# Slow motion factorial_iter(5):
# acc=1, n=5 → acc=5,  n=4
#             → acc=20, n=3
#             → acc=60, n=2
#             → acc=120,n=1 → exit → return 120

print("Factorial comparison:")
for n in [0, 1, 5, 10]:
    t = factorial_tail(n)
    i = factorial_iter(n)
    match = "✓" if t == i else "✗"
    print(f"  {n}! = {i}  {match}")

# ── Recursion limit demo ──────────────────────────────────────────────────────
print(f"\nRecursion limit: {sys.getrecursionlimit()}")

# This would crash: factorial_tail(2000) → RecursionError
# The iterative version handles it fine:
print(f"factorial_iter(2000) has {len(str(factorial_iter(2000)))} digits — no stack issue")

# ── Reverse string: recursive vs iterative ────────────────────────────────────
def reverse_recursive(s: str) -> str:
    """LC 344 — Reverse String (recursive, O(n) stack)."""
    if len(s) <= 1:
        return s
    return reverse_recursive(s[1:]) + s[0]  # combine: rest reversed + first char

def reverse_iter(s: str) -> str:
    """LC 344 — Reverse String (iterative, O(1) extra, O(n) output)."""
    return s[::-1]   # Python slice — clean and fast

print(reverse_recursive("hello"))   # olleh
print(reverse_iter("hello"))        # olleh

# Simplicity and clarity is Gold


<a id='7'></a>

## 7. 🧩 Pattern 4: Memoization (Top-Down DP) — LC 70, 198, 322

---

```
PROBLEM:  Recursive solution recomputes the same sub-problems. Add a cache.

APPROACH: Before computing, check if result is already cached.
          After computing, store result in cache before returning.

MEMOIZATION TEMPLATE:
  cache = {}
  def f(n):
      if n in cache: return cache[n]    # cache hit — O(1) lookup
      result = ... f(n-1) ... f(n-2) ...
      cache[n] = result                 # store before return
      return result

  OR use @lru_cache(maxsize=None) — same thing, cleaner syntax.

SLOW MOTION TRACE on coin_change([1,5,11], amount=15):
  f(15):
    f(14): f(13): f(12): f(11): f(10): f(9): f(8): f(7): f(6): f(5): f(4): f(3): f(2): f(1): f(0)=0
    Each computed ONCE — stored in memo.
    f(10) = 1 + f(5) — f(5) already in memo!

  Without memo: exponential.  With memo: O(amount * len(coins)).

BOTTOM-UP DP (tabulation) vs TOP-DOWN (memoization):
  Top-down: natural recursion + cache. Easier to write.
  Bottom-up: iterative, fills table from smallest sub-problem up. More efficient (no stack).
  Both have the same O() complexity. Pick top-down for interviews first.

TIME / SPACE: depends on problem — generally O(state_count) time + space.
```


In [ ]:
from functools import lru_cache
from typing import List

# ── LC 70 — Climb Stairs (already seen, reinforcement) ────────────────────────
@lru_cache(maxsize=None)
def climbStairs(n: int) -> int:
    """
    LC 70 — Climbing Stairs. fib pattern with memoization.
    Time: O(n)  Space: O(n)
    """
    if n <= 2: return n
    return climbStairs(n-1) + climbStairs(n-2)

# ── LC 198 — House Robber ─────────────────────────────────────────────────────
def rob(nums: List[int]) -> int:
    """
    LC 198 — House Robber.
    Approach: memoized recursion — at each house, rob or skip.
    Args:
        nums (List[int]): money in each house.
    Returns:
        int: maximum money without robbing adjacent houses.
    Time:  O(n)  Space: O(n) memo + O(n) stack
    """
    memo = {}
    def dp(i: int) -> int:
        if i >= len(nums):
            return 0                         # past the last house — no money
        if i in memo:
            return memo[i]                   # already solved this house
        rob_this  = nums[i] + dp(i + 2)     # rob house i, skip neighbor
        skip_this = dp(i + 1)               # skip house i, decide at i+1
        memo[i] = max(rob_this, skip_this)
        return memo[i]
    return dp(0)

# Slow motion on [2,7,9,3,1]:
# dp(0): rob=2+dp(2), skip=dp(1)
#   dp(2): rob=9+dp(4), skip=dp(3)
#     dp(4): rob=1+dp(6)=1+0=1, skip=dp(5)=0 → max(1,0)=1   memo[4]=1
#     dp(3): rob=3+dp(5)=3, skip=dp(4)=1 → max(3,1)=3       memo[3]=3
#   dp(2): rob=9+1=10, skip=3 → 10   memo[2]=10
#   dp(1): rob=7+dp(3)=7+3=10, skip=dp(2)=10 → 10  memo[1]=10
# dp(0): rob=2+10=12, skip=10 → 12  ✓

# ── LC 322 — Coin Change ──────────────────────────────────────────────────────
def coinChange(coins: List[int], amount: int) -> int:
    """
    LC 322 — Coin Change.
    Approach: memoized recursion — at each amount, try each coin.
    Args:
        coins (List[int]): available denominations.
        amount (int): target amount.
    Returns:
        int: fewest coins to make amount, or -1 if impossible.
    Time:  O(amount * len(coins))  Space: O(amount)
    """
    memo = {}
    def dp(rem: int) -> int:
        if rem == 0: return 0             # base: made exact change
        if rem < 0:  return float('inf')  # overshot — invalid path
        if rem in memo: return memo[rem]
        best = min(dp(rem - c) for c in coins)  # try each coin
        memo[rem] = 1 + best if best != float('inf') else float('inf')
        return memo[rem]
    result = dp(amount)
    return result if result != float('inf') else -1

def test_harness(fn_rob, fn_coin):
    rob_tests = [([2,7,9,3,1], 12), ([1,2,3,1], 4), ([0], 0)]
    for *inputs, expected in rob_tests:
        got = fn_rob(*inputs)
        print(f"rob({inputs[0]}) = {got}  {'✓' if got==expected else '✗'}")

    coin_tests = [([1,5,11], 15, 3), ([2], 3, -1), ([1,2,5], 11, 3)]
    for *inputs, expected in coin_tests:
        got = fn_coin(*inputs)
        print(f"coinChange({inputs[0]}, {inputs[1]}) = {got}  {'✓' if got==expected else '✗'}")

test_harness(rob, coinChange)
print("climbStairs, rob, coinChange defined.")

# Simplicity and clarity is Gold


<a id='8'></a>

## 8. 🧩 Pattern 5: Backtracking = Recursion + Undo — LC 46, 78

---

```
PROBLEM:  Generate ALL solutions (permutations, subsets, paths). At each step,
          try a choice, recurse, then UNDO the choice before trying the next.

TEMPLATE:
  def backtrack(state, choices):
      if is_complete(state):
          results.append(copy(state))    # ← COPY, not reference!
          return
      for choice in choices:
          if is_valid(choice, state):
              state.append(choice)       # MAKE the choice
              backtrack(state, ...)      # recurse
              state.pop()               # UNDO the choice ← this is the "back" in backtrack

SLOW MOTION TRACE — permutations([1,2,3]):
  backtrack([], [1,2,3])
    choose 1: backtrack([1], [2,3])
      choose 2: backtrack([1,2], [3])
        choose 3: backtrack([1,2,3], []) → add [1,2,3] ✓
                                        ← undo 3, [1,2]
      ← undo 2, [1]
      choose 3: backtrack([1,3], [2])
        choose 2: backtrack([1,3,2], []) → add [1,3,2] ✓
                                        ← undo 2, [1,3]
      ← undo 3, [1]
    ← undo 1, []
    choose 2: ...

KEY INSIGHT: The undo step is what makes backtracking work — it restores the state
             so sibling branches see a clean slate. Missing the undo is the #1 bug.

TIME: O(n! * n) for permutations — you must generate all of them.
```


In [ ]:
from typing import List

def permute(nums: List[int]) -> List[List[int]]:
    """
    LC 46 — Permutations (backtracking).
    Approach: build each permutation by choosing unused elements, undo each choice.
    Args:
        nums (List[int]): distinct integers.
    Returns:
        List[List[int]]: all n! permutations.
    Time:  O(n! * n) — n! permutations, each of length n (copy cost)
    Space: O(n) — call stack depth + current path
    """
    results = []

    def backtrack(current: List[int], remaining: List[int]):
        if not remaining:               # all elements placed — complete permutation
            results.append(list(current))  # COPY — don't append a reference!
            return
        for i in range(len(remaining)):
            current.append(remaining[i])                # MAKE choice: add remaining[i]
            backtrack(current, remaining[:i] + remaining[i+1:])  # recurse without chosen
            current.pop()                               # UNDO choice

    backtrack([], nums)
    return results

# Slow motion on [1,2]:
# backtrack([], [1,2])
#   i=0: current=[1], remaining=[2]
#     backtrack([1], [2])
#       i=0: current=[1,2], remaining=[]
#         → add [1,2]  pop → current=[1]
#   pop → current=[]
#   i=1: current=[2], remaining=[1]
#     backtrack([2], [1])
#       i=0: current=[2,1], remaining=[]
#         → add [2,1]  pop → current=[2]
#   pop → current=[]
# results = [[1,2], [2,1]] ✓

def subsets(nums: List[int]) -> List[List[int]]:
    """
    LC 78 — Subsets (backtracking).
    Approach: at each element, choose to include or exclude.
    Args:
        nums (List[int]): distinct integers.
    Returns:
        List[List[int]]: 2^n subsets including empty set.
    Time:  O(2^n * n)  Space: O(n) stack
    """
    results = []

    def backtrack(start: int, current: List[int]):
        results.append(list(current))   # add current subset at every state
        for i in range(start, len(nums)):
            current.append(nums[i])             # include nums[i]
            backtrack(i + 1, current)           # recurse with remaining elements
            current.pop()                        # exclude nums[i] (undo)

    backtrack(0, [])
    return results

def test_harness_perm(fn):
    tests = [
        ([1, 2, 3], 6),   # 3! = 6 permutations
        ([0, 1], 2),
        ([1], 1),
    ]
    passed = 0
    for *inputs, expected_count in tests:
        got = fn(*inputs)
        ok = len(got) == expected_count and len(set(tuple(p) for p in got)) == expected_count
        print(f"permute({inputs[0]}): {len(got)} perms  {'✓' if ok else '✗'}")
        passed += ok
    print(f"{passed}/{len(tests)} tests passed (permute)")

def test_harness_sub(fn):
    tests = [
        ([1,2,3], 8),   # 2^3 = 8 subsets
        ([0], 2),       # [], [0]
    ]
    passed = 0
    for *inputs, expected_count in tests:
        got = fn(*inputs)
        ok = len(got) == expected_count
        print(f"subsets({inputs[0]}): {len(got)} subsets  {'✓' if ok else '✗'}")
        passed += ok
    print(f"{passed}/{len(tests)} tests passed (subsets)")

test_harness_perm(permute)
test_harness_sub(subsets)
print("permute, subsets defined.")

# Simplicity and clarity is Gold


<a id='9'></a>

## 9. 🧩 Pattern 6: Divide and Conquer — LC 912, 148

---

```
PROBLEM:  Problem can be solved by: split into smaller pieces → solve each →
          combine answers.

TEMPLATE:
  def solve(problem):
      if is_base_case(problem):
          return base_case_answer
      left, right = split(problem)     # divide
      left_ans    = solve(left)        # conquer left
      right_ans   = solve(right)       # conquer right
      return combine(left_ans, right_ans)  # combine

EXAMPLES:
  merge_sort: split=halve, conquer=sort each half, combine=merge
  binary_search: split=discard half, conquer=search remaining half, combine=return
  max_subarray: split=halve, conquer=max in each half+crossing, combine=take max

DIVIDE vs BACKTRACK:
  Divide & conquer: each sub-problem is INDEPENDENT — no shared state
  Backtracking: sub-problems share state (the current path) — must undo

KEY INSIGHT: D&C recursion trees are balanced → O(log n) depth → O(n log n) total
             if each level does O(n) work (like merge sort's merge step).

TIME: use the Master Theorem:
  T(n) = aT(n/b) + f(n)
  merge sort: T(n) = 2T(n/2) + O(n)  → O(n log n)
  binary search: T(n) = T(n/2) + O(1) → O(log n)
```


<a id='10'></a>

## 10. Common Pitfalls

```
PITFALL 1: Missing base case
  def bad(n): return bad(n-1) + bad(n+1)   ← no base case → infinite recursion

PITFALL 2: Not reducing the problem
  def bad(n):
      if n == 0: return 0
      return bad(n)    ← n doesn't change → infinite recursion

PITFALL 3: Missing return statement
  def bad(n):
      if n == 0: return 0
      bad(n-1)         ← forgot return → all calls return None

PITFALL 4: Mutating shared mutable argument
  def bad(nums, path=[]):    ← mutable default arg is shared across calls!
      path.append(1)
      ...

  CORRECT:
  def good(nums, path=None):
      if path is None: path = []    ← create new list each call

PITFALL 5: Appending reference instead of copy
  results.append(current)   ← current will be mutated later!
  CORRECT:
  results.append(list(current))   ← copy it first

PITFALL 6: Assuming Python optimizes tail recursion
  Python NEVER does TCO. Tail recursion still uses O(n) stack.
  Always convert to iteration for n > ~900.

PITFALL 7: Off-by-one in base case
  if n == 0: return 0    ← what about n = 1?
  if n <= 1: return n    ← safer for fib-style problems
```


In [ ]:
# ── Pitfall demos — each is a live fix ────────────────────────────────────────

# PITFALL 3: Missing return
def wrong_sum(n):
    if n == 0: return 0
    wrong_sum(n-1) + n    # forgot return!

def correct_sum(n):
    if n == 0: return 0
    return correct_sum(n-1) + n   # return the result!

print(f"wrong_sum(5)   = {wrong_sum(5)}")    # None  ← silent bug
print(f"correct_sum(5) = {correct_sum(5)}")  # 15    ✓

print()

# PITFALL 4: Mutable default argument
def bad_collect(n, result=[]):
    result.append(n)
    if n == 0: return result
    return bad_collect(n-1, result)

def good_collect(n, result=None):
    if result is None: result = []    # new list every top-level call
    result.append(n)
    if n == 0: return result
    return good_collect(n-1, result)

call1 = bad_collect(2)
call2 = bad_collect(2)   # result from call1 leaks into call2!
print(f"bad_collect  call1={call1}  call2={call2}")  # call2 has extra items!

good1 = good_collect(2)
good2 = good_collect(2)
print(f"good_collect call1={good1}  call2={good2}")  # independent ✓

print()

# PITFALL 5: Appending reference
def bad_permute(nums):
    results = []
    path = []
    def bt(remaining):
        if not remaining:
            results.append(path)     # BUG: appends same list reference!
        for i in range(len(remaining)):
            path.append(remaining[i])
            bt(remaining[:i] + remaining[i+1:])
            path.pop()
    bt(nums)
    return results

def good_permute(nums):
    results = []
    path = []
    def bt(remaining):
        if not remaining:
            results.append(list(path))  # COPY: each append is independent ✓
        for i in range(len(remaining)):
            path.append(remaining[i])
            bt(remaining[:i] + remaining[i+1:])
            path.pop()
    bt(nums)
    return results

bad  = bad_permute([1,2])
good = good_permute([1,2])
print(f"bad_permute([1,2])  = {bad}")    # all empty! reference was cleared
print(f"good_permute([1,2]) = {good}")   # [[1,2],[2,1]] ✓

# Simplicity and clarity is Gold


<a id='11'></a>

## 11. 🧩 Pattern 7: LC 206 — Reverse Linked List (Recursive)

---

```
PROBLEM:  Reverse a singly linked list. Return new head.
          Iterative is O(1) space. Recursive is O(n) stack — important to know both.

RECURSIVE APPROACH:
  "If I reverse everything after head, and then point head.next.next at head, I'm done."

SLOW MOTION TRACE on [1→2→3→4→5]:
  reverse(1): tail = reverse(2→3→4→5)
    reverse(2): tail = reverse(3→4→5)
      reverse(3): tail = reverse(4→5)
        reverse(4): tail = reverse(5)
          reverse(5): base case → return 5 (new head)
        ← tail=5. 5.next=4. 4.next=None. return 5
      ← tail=5. 4.next=3. 3.next=None. return 5
    ← tail=5. 3.next=2. 2.next=None. return 5
  ← tail=5. 2.next=1. 1.next=None. return 5

  Result: 5→4→3→2→1 ✓

KEY INSIGHT: node.next.next = node  flips the pointer.
             node.next = None  severs the original forward pointer (CRITICAL — avoid cycle).

TIME / SPACE: O(n) / O(n) stack — use iterative (O(1) space) for production.
```


In [ ]:
from typing import Optional

class ListNode:
    def __init__(self, val=0, nxt=None):
        self.val = val
        self.next = nxt
    def __repr__(self):
        vals = []
        cur = self
        while cur:
            vals.append(str(cur.val))
            cur = cur.next
        return "->".join(vals)

def reverseList_recursive(head: Optional[ListNode]) -> Optional[ListNode]:
    """
    LC 206 — Reverse Linked List (recursive).
    Approach: reverse tail, then repoint head.next.next to head.
    Args:
        head (Optional[ListNode]): head of linked list.
    Returns:
        Optional[ListNode]: new head (old tail).
    Time:  O(n) — visit every node once
    Space: O(n) — recursion stack n frames deep
    """
    if not head or not head.next:
        return head                        # base case: 0 or 1 node — already reversed

    new_head = reverseList_recursive(head.next)  # reverse the rest; new_head = old tail
    head.next.next = head              # the node after head now points BACK to head
    head.next = None                   # sever head's forward link — prevent cycle!
    return new_head                    # new_head is the same old tail throughout

def reverseList_iterative(head: Optional[ListNode]) -> Optional[ListNode]:
    """
    LC 206 — Reverse Linked List (iterative). O(n) / O(1).
    Approach: walk forward, flipping each .next pointer.
    """
    prev, curr = None, head
    while curr:
        nxt = curr.next        # save next before we overwrite it
        curr.next = prev       # flip pointer backward
        prev = curr            # advance prev
        curr = nxt             # advance curr
    return prev                # prev is the new head (last node we processed)

# Build test list [1→2→3→4→5]
def build(vals):
    dummy = ListNode(0)
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

lst1 = build([1,2,3,4,5])
lst2 = build([1,2,3,4,5])
print(f"original:   {lst1}")
print(f"recursive:  {reverseList_recursive(lst1)}")   # 5->4->3->2->1
print(f"iterative:  {reverseList_iterative(lst2)}")   # 5->4->3->2->1

# Simplicity and clarity is Gold


<a id='12'></a>

## 12. 🧩 Pattern 8: LC 104 / LC 112 — Tree Depth & Path Sum

---

```
PROBLEM:  LC 104: Maximum depth of binary tree (already traced above).
          LC 112: Does a root-to-leaf path sum equal targetSum?

PATH SUM APPROACH:
  "Subtract node.val from targetSum. At a leaf, check if remainder == 0."

SLOW MOTION TRACE on targetSum=22:
     5
    / \
   4   8
  /   / \
 11  13   4
 / \       \
7   2       1

  hasPathSum(5, 22): recurse(4, 17) OR recurse(8, 17)
  hasPathSum(4, 17): recurse(11, 13) OR recurse(None, 13)→False
  hasPathSum(11, 13): recurse(7, 6) OR recurse(2, 11)
  hasPathSum(7, 6): leaf. 6 ≠ 0 → False
  hasPathSum(2, 11): leaf. 11 ≠ 0 → False
  hasPathSum(8, 17): recurse(13, 4) OR recurse(4, 13)
  hasPathSum(13, 4): leaf. 4 ≠ 0 → False
  hasPathSum(4, 13): recurse(None)→False OR recurse(1, 9)
  hasPathSum(1, 9): leaf. 9 ≠ 0 → False
  All False → return False ✓ (no path sums to 22 in this shape)

KEY INSIGHT: Tree recursion = "compute at leaves, combine upward."
             OR semantics: any path works → return left OR right.
             AND semantics: all paths must work → return left AND right.
```


In [ ]:
from typing import Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def hasPathSum(root: Optional[TreeNode], targetSum: int) -> bool:
    """
    LC 112 — Path Sum.
    Approach: subtract node.val, recurse; at leaf check if remainder == 0.
    Args:
        root (Optional[TreeNode]): root of binary tree.
        targetSum (int): target path sum.
    Returns:
        bool: True if any root-to-leaf path has sum == targetSum.
    Time:  O(n)  Space: O(h) stack
    """
    if not root:
        return False                          # fell off the tree — no path here
    remaining = targetSum - root.val
    if not root.left and not root.right:      # at a leaf
        return remaining == 0                 # did this path hit the target?
    return hasPathSum(root.left, remaining) or hasPathSum(root.right, remaining)

def maxDepth(root: Optional[TreeNode]) -> int:
    """LC 104 — already shown, included for test completeness."""
    if not root: return 0
    return 1 + max(maxDepth(root.left), maxDepth(root.right))

# Build test tree
#      5
#     / \
#    4   8
#   /   /
#  11  13
#  /\
# 7  2
root = TreeNode(5,
    TreeNode(4, TreeNode(11, TreeNode(7), TreeNode(2)), None),
    TreeNode(8, TreeNode(13), TreeNode(4, None, TreeNode(1))))

print(f"maxDepth     = {maxDepth(root)}")             # 4
print(f"hasPathSum(22) = {hasPathSum(root, 22)}")     # False for this tree shape (see trace)
print(f"hasPathSum(26) = {hasPathSum(root, 26)}")     # True: 5→8→13

# Build simple tree for basic path sum test
#   1
#  / \
# 2   3
simple = TreeNode(1, TreeNode(2), TreeNode(3))
print(f"simple hasPathSum(3) = {hasPathSum(simple, 3)}")   # True: 1→2

# Simplicity and clarity is Gold


<a id='13'></a>

## 13. Full Decision Map

```
QUESTION TYPE                            PATTERN               LC PROBLEMS
──────────────────────────────────────────────────────────────────────────────
Compute n-th value using n-1 and n-2     Linear recursion       70, 509
Traverse or operate on tree/graph        Tree recursion         104, 112, 226
Minimize/maximize over choices with reuse Memoized recursion    70, 198, 322
Generate all valid sequences/subsets     Backtracking           46, 78, 39, 51
Sort / find: divide array in half        Divide & conquer       912, 148
Reverse / reorder linked list            Linear recursion       206, 25
Validate tree property                   Tree recursion (AND)   98, 110
Accumulate without stack (tail style)    Iterative (not tail!)  factorial
Deep recursion (n > 900)                 Convert to iteration   —
──────────────────────────────────────────────────────────────────────────────
```


<a id='14'></a>

## 14. Interview Cheat Sheet

### 1. When to reach for recursion:

| Signal | What to Do |
|--------|------------|
| Tree / graph problem | Tree recursion (DFS) |
| "all permutations / subsets / paths" | Backtracking |
| "divide array in half, combine" | Divide & conquer |
| "solve n using n-1" | Linear recursion + memoize if overlapping |
| "minimum coins / stairs / jumps" | Memoized recursion / DP |

### 2. The universal recursion template:

```python
def solve(state):
    # 1. BASE CASE first
    if base_condition(state):
        return base_value

    # 2. REDUCE the problem
    smaller = make_smaller(state)

    # 3. TRUST the recursion
    sub_result = solve(smaller)

    # 4. COMBINE
    return combine(sub_result, state)
```

### 3. Backtracking template:

```python
results = []
def backtrack(current, remaining):
    if not remaining:
        results.append(list(current))   # COPY — never append reference
        return
    for choice in remaining:
        current.append(choice)          # MAKE choice
        backtrack(current, without(choice, remaining))
        current.pop()                   # UNDO choice
backtrack([], choices)
```

### 4. Gotchas to not forget:

```
❌  Forget to return the recursive result → silent None bug
❌  Mutable default argument def f(n, lst=[]) → shared across calls
❌  Append reference instead of copy → all results mutated to []
❌  Expect Python to do TCO → it never does, use iterative for n>900
✅  Base case: always test n=0, n=1, empty input
✅  Every recursive call must make the problem strictly SMALLER
✅  @lru_cache replaces memo dict — same O(), cleaner code
✅  Backtracking: undo must exactly mirror the make-choice step
```


```
RECURSION FUNDAMENTALS MASTER MAP
════════════════════════════════════════════════════

                   RECURSION
                       │
    ┌──────────────────┼──────────────────┐
    ▼                  ▼                  ▼
LINEAR             TREE              GENERATE ALL
RECURSION          RECURSION         (Backtracking)
    │                  │                  │
fib, factorial     binary tree        permutations
reverse list       graph DFS          subsets
linked list        max depth          N-queens
O(n) stack         O(h) stack         O(n!) output
    │                  │
    ▼                  ▼
MEMOIZE?        WHAT TO RETURN
    │            AT BASE CASE?
    YES →        ─────────────
  lru_cache      depth:  0
  or memo {}     valid:  True
    │             path:  []
    ▼             found: False
TOP-DOWN DP
O(states) time
O(states) space
    │
    ▼
CONVERT TO
ITERATIVE
if n > 900
O(1) space
```

---
*End of Recursion Fundamentals Master Guide — Sean Edition*
